# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [2]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.svm import SVC
import itertools
from tqdm.notebook import tqdm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import numpy as np

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [3]:
class FeatureExtractor:
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        required_cols = {'uid', 'labname', 'numTrials', 'timestamp'}
        missing = required_cols - set(X.columns)
        if missing:
            raise ValueError(f'Missing columns: {missing}')
        
        df = X.copy()

        if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
            df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.copy()

        df['hour'] = df.timestamp.dt.hour
        df['weekday'] = df.timestamp.dt.dayofweek

        return df.drop('timestamp', axis=1)

In [4]:
class MyOneHotEncoder:
    def __init__(self, target=None):
        self.target = target
        self.encoder = None
        self.categorical_columns = None
        

    def fit(self, X, y=None):
        self.categorical_columns = X.drop(self.target, axis=1).select_dtypes(include=['object', 'category']).columns.tolist()

        if self.categorical_columns:
            self.encoder = OneHotEncoder(
                sparse=False,
                handle_unknown='ignore'
            )
            self.encoder.fit(X[self.categorical_columns])

        return self

    def transform(self, X):
        df = X.copy()

        if not self.categorical_columns:
            return df

        encoded_features = self.encoder.transform(df[self.categorical_columns])
        encoded_cols = self.encoder.get_feature_names(self.categorical_columns)
        encoded_df = pd.DataFrame(encoded_features, columns=encoded_cols, index=df.index)
        
        result_df = pd.concat(
            [df.drop(columns=self.categorical_columns), encoded_df],
            axis=1
        )

        return result_df
    
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

In [5]:
class TrainValidationTest:
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def split(self):
        X_train, X_test, y_train, y_test = train_test_split(self.X, self.y, test_size=0.2, random_state=21, stratify=self.y)
        X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [6]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict

    def choose(self, X_train, y_train, X_valid, y_valid):
        results = []
        for i, grid in enumerate(self.grids):
            model_name = self.grid_dict[i]
            param_grid = grid.param_grid
            keys = param_grid.keys()
            values = param_grid.values()
            model_class = grid.estimator
            combinations = list(itertools.product(*values))

            model_results = []
            print(f'Estimator: {model_name}')
            for combination in tqdm(combinations):
                params = dict(zip(keys, combination))
                model = model_class(**params)
                model.fit(X_train, y_train)
                accuracy = model.score(X_valid, y_valid)

                model_results.append({
                    'model': model_name,
                    'params': params,
                    'training_score': model.score(X_train, y_train),
                    'valid_score': accuracy
                })  

            best = sorted(model_results, key=lambda x: x['training_score'], reverse=True)[0]
            print(f'Best params: {best["params"]}')
            print(f'Best training accuracy: {best["training_score"]:.5f}')
            print(f'Validation set accuracy score for best params: {best["valid_score"]:.5f}')
            results.append(best)

        self.grid_search_df = pd.DataFrame(results).drop('training_score', axis=1).sort_values('valid_score', ascending=False)
        return self.grid_search_df['model'].head(1)
    
    def best_results(self):
        return self.grid_search_df


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [7]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        score = self.estimator.score(X_test, y_test)
        print(f'Accuracy of the final model {score}')
        return score
    
    def save_model(self, path):
        return joblib.dump(self.estimator, path)

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [8]:
df = pd.read_csv('../data/checker_submits.csv', parse_dates=['timestamp'])

df.head()

,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104


In [9]:
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder(target='weekday'))])

data = preprocessing.fit_transform(df)

In [10]:
X = data.drop('weekday', axis=1)
y = data.weekday

In [11]:
X_train, X_valid, X_test, y_train, y_valid, y_test = TrainValidationTest(X, y).split()

In [12]:
svm_params = {
    'kernel': ['linear', 'rbf', 'sigmoid'],
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced', None],
    'random_state': [21]
}

dt_params = {
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

rf_params = {
    'n_estimators': [5, 10, 50, 100],
    'max_depth': np.arange(1, 50),
    'class_weight': ['balanced', None],
    'criterion': ['entropy', 'gini'],
    'random_state': [21]
}

In [13]:
gs_svm = GridSearchCV(estimator=SVC, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_tree = GridSearchCV(estimator=DecisionTreeClassifier, param_grid=dt_params, scoring='accuracy', cv=2, n_jobs=-1)
gs_rf = GridSearchCV(estimator=RandomForestClassifier, param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=-1)

In [14]:
grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {
    0: 'SVM',
    1: 'Decision Tree',
    2: 'Random Forest'
}

In [15]:
model_selection = ModelSelection(grids, grid_dict)

In [16]:
model_selection.choose(X_train, y_train, X_valid, y_valid)

Estimator: SVM


  0%|          | 0/72 [00:00<?, ?it/s]

Best params: {'kernel': 'rbf', 'C': 10, 'gamma': 'auto', 'class_weight': None, 'random_state': 21}
Best training accuracy: 0.95176
Validation set accuracy score for best params: 0.87778
Estimator: Decision Tree


  0%|          | 0/196 [00:00<?, ?it/s]

Best params: {'max_depth': 21, 'class_weight': 'balanced', 'criterion': 'entropy', 'random_state': 21}
Best training accuracy: 1.00000
Validation set accuracy score for best params: 0.85556
Estimator: Random Forest


  0%|          | 0/784 [00:00<?, ?it/s]

Best params: {'n_estimators': 50, 'max_depth': 19, 'class_weight': 'balanced', 'criterion': 'entropy', 'random_state': 21}
Best training accuracy: 1.00000
Validation set accuracy score for best params: 0.89259


2    Random Forest
Name: model, dtype: object

In [17]:
model_selection.best_results()

,model,params,valid_score
2,Random Forest,"{'n_estimators': 50, 'max_depth': 19, 'class_w...",0.892593
0,SVM,"{'kernel': 'rbf', 'C': 10, 'gamma': 'auto', 'c...",0.877778
1,Decision Tree,"{'max_depth': 21, 'class_weight': 'balanced', ...",0.855556


In [18]:
final = Finalize(RandomForestClassifier(n_estimators=50, max_depth=18, class_weight='balanced', criterion='entropy', random_state=21))

In [19]:
accuracy = final.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model 0.8994082840236687


In [20]:
path = f'../data/random_forest{accuracy:.5f}.sav'

final.save_model(path)

['../data/random_forest0.89941.sav']